# PSF-Zero: Unitary Synthesis with Geometric Optimization
This notebook demonstrates how to use the `PSFHybridSynthesizer` to compile arbitrary 2-qubit unitaries into native `RZZ` and local rotation gates. 
By utilizing analytic subgradients for L1/TV penalties and an Adam-based optimizer with curvature-aware clipping, PSF-Zero synthesizes circuits that inherently minimize pulse dissipation and parameter drift.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer

# Import PSF-Zero core (Ensure psf_synthesis.py is in the same directory)
from psf_synthesis import PSFHyper, PSFHybridSynthesizer, compose_circuit_unitary, F_avg

# Set plotting style
plt.style.use('dark_background')

In [ ]:
# 1. Generate a complex, random 2-Qubit Target Unitary
rng = np.random.default_rng(42)
Q, _ = np.linalg.qr(rng.normal(size=(4,4)) + 1j*rng.normal(size=(4,4)))
U_target = Q @ np.diag(np.exp(1j*rng.normal(size=4))) @ Q.conj().T

# 2. Initialize the Synthesizer
hyper = PSFHyper(m=3, iters=150, lr=0.25, alpha_proj=1e-2, beta_H=5e-3, beta_TV=5e-3)
synth = PSFHybridSynthesizer(hyper)

# 3. Track metrics for visualization
history_f_avg = []
history_dissipation = []

lr0 = hyper.lr
for t in range(hyper.iters):
    # Cosine annealing LR
    lr = max(1e-4, lr0 * (0.5 * (1 + np.cos(np.pi * t / hyper.iters))))
    
    # Calculate analytic gradients
    grad = synth._ps_grad(U_target)
    
    # Geometrically clamped Adam step
    step = synth._adam_step(grad, lr)
    
    # Update state
    vec = synth._flat()
    vec -= step
    synth._set_flat(vec)
    
    # Record metrics every iteration
    U_current = compose_circuit_unitary(synth.params_angles, synth.taus)
    f_val = F_avg(U_current, U_target, polar_fix=False)
    dissipation = np.sum(np.abs(synth.params_angles)) + np.sum(np.abs(synth.taus))
    
    history_f_avg.append(f_val)
    history_dissipation.append(dissipation)

# 4. Plot the "Geometric Surrender" (Convergence)
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = '#00ffcc'
ax1.set_xlabel('Optimization Iterations (Phase Space)')
ax1.set_ylabel('Average Gate Fidelity ($F_{avg}$)', color=color1)
ax1.plot(history_f_avg, color=color1, linewidth=2, label='Fidelity (↑)')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0.4, 1.01)

ax2 = ax1.twinx()  
color2 = '#ff3366'
ax2.set_ylabel('Pulse Dissipation (L1 Norm)', color=color2)  
ax2.plot(history_dissipation, color=color2, linestyle='--', linewidth=2, label='Dissipation (↓)')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('PSF-Zero Convergence: Maximizing Fidelity while Minimizing Entropy')
fig.tight_layout()  
plt.show()

print(f"Final Fidelity: {history_f_avg[-1]:.6f}")

In [ ]:
# Extract the resulting highly-optimized Qiskit circuit
qc_synthesized = synth.as_qiskit()
print("Synthesized Low-Dissipation Circuit:")
display(qc_synthesized.draw('mpl', style='iqp-dark'))